In [1]:
# ============ CONFIG ============
INPUT_DIR   = "/kaggle/input/datasets/newmailserver/moge-d2p-output/conference_room"          # Path to dataset attached to notebook
WORK_DIR    = "/kaggle/working/SPAG4d"                                        # Where SPAG4d repo gets cloned
OUTPUT_DIR  = "/kaggle/working/output"                                        # Root output folder
UPLOAD_DIR  = "/kaggle/temp/upload"                                           # Staging dir containing dataset to upload

SPAG_REPO   = "https://github.com/cedarconnor/SPAG4d.git"

# Image extensions to pick up from each folder in INPUT_DIR
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")

# GPU assignment — Kaggle dual T4 (cuda:0, cuda:1)
GPU_IDS = [0, 1]

# Kaggle Dataset Upload Config
KAGGLE_USERNAME = "newmailserver"
DATASET_SLUG    = "moge-splat-output"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("✅ Config loaded.")
print("Input dir :", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)


✅ Config loaded.
Input dir : /kaggle/input/datasets/newmailserver/moge-d2p-output/conference_room
Output dir: /kaggle/working/output


In [2]:
# ============ INSTALL & SYNC ============
!pip install -q uv kagglehub

%cd {WORK_DIR}/..
!rm -rf {WORK_DIR}
!git clone --depth 1 {SPAG_REPO} {WORK_DIR}

%cd {WORK_DIR}
!uv sync
!uv pip install opencv-python Pillow numpy torch scipy plyfile imageio[pyav]

import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
%env MPLBACKEND=Agg

print("✅ SPAG4d installed and synced via uv.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 74.4 MB/s eta 0:00:00:00:0100:01
/kaggle/working
Cloning into '/kaggle/working/SPAG4d'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (313/313), done.
remote: Compressing objects: 100% (289/289), done.
remote: Total 313 (delta 9), reused 278 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (313/313), 64.00 MiB | 45.83 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/kaggle/working/SPAG4d
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 101 packages in 1.55s                                       
Prepared 62 packages in 32.29s                                           
░░░░░░░░░░░░░░░░░░░░ [0/63] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not b

In [3]:
# ============ SPAG-4D CONFIGURATION ============
# Set any value manually, or leave as None to let SPAG-4D auto-calculate from scene analysis
# ============ INDOOR-OPTIMIZED SPAG-4D CONFIGURATION ============
SPAG_CONFIG = {
    # Density & Resolution
    "stride": 1,                 # 1 for full resolution & dense coverage (use 2 if VRAM constrained)
    "depth_min": 0.01,           # Don't drop close-up objects (table edges, chairs)
    "depth_max": 500.0,          # Large bound so no room walls are cut off
    
    # Disable Sky Filtering Completely for Indoor Rooms
    "sky_detection": "none",     # "none" stops SPAG from deleting distant walls/windows
    "sky_threshold": 0.0,        # 0.0 disables sky depth cutoff
    "sky_mode": "skip",
    
    # Disable Geometry Pruning (Prevents deleting thin furniture & corners)
    "outlier_pruning": 0.0,      # 0.0 = Keep all chair legs, fixtures, and room corners
    "grazing_angle": 0.0,        # 0.0 = Keep edge-on surfaces (walls & floors)
    "sparse_pruning": 0.0,       # 0.0 = Keep thin structures
    
    # Full Ceiling, Floor & Wall Coverage
    "pole_thinning": False,      # False = Complete ceiling & floor reconstruction
    "min_density_ratio": 1.0,    # Uniform density across all angles
    "default_opacity": 0.98,     # Solid, opaque splats
    "disc_thickness": 0.20,      # Thicker discs to prevent see-through walls
    "global_scale": 1.20,        # 1.15 - 1.25 gives seamless splat overlap without gaps
    "fill_depth_holes": True,    # Inpaint specular/zero depth holes (tables, glass, screens)
}



# ============ INFERENCE MANAGER USING NATIVE SPAG4D ============
import sys, os, glob, shutil, time, queue, threading
import cv2, numpy as np, torch
from PIL import Image, ImageOps

# Add SPAG4d to Python path
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Import directly from native SPAG4d package
from spag4d.spag_converter import depth_to_gaussians, SPAGParams
from spag4d.scene_analysis import compute_scene_defaults
from spag4d.scene_filter import (
    prune_outliers,
    prune_grazing_angle,
    prune_sparse_regions,
    apply_sky_mode_to_gaussians,
    SkyMode,
)
from spag4d.ply_writer import save_ply_gsplat

# 1. Discover all scenes (folders with depth.exr and image)
def find_scenes(input_root):
    depth_files = glob.glob(os.path.join(input_root, "**", "depth.exr"), recursive=True)
    depth_files = [d for d in depth_files if f"{os.sep}splitted{os.sep}" not in d]
    
    scenes = []
    for d_path in sorted(depth_files):
        folder = os.path.dirname(d_path)
        folder_name = os.path.basename(folder)

        candidates = []
        for ext in IMAGE_EXTS:
            candidates.extend(glob.glob(os.path.join(folder, f"*{ext}")))
            candidates.extend(glob.glob(os.path.join(folder, f"*{ext.upper()}")))

        valid_imgs = [
            img for img in candidates
            if not any(k in os.path.basename(img).lower() for k in ["depth_vis", "normal_vis", "mask"])
        ]
        
        # Check for optional normal map in folder
        norm_candidates = glob.glob(os.path.join(folder, "*normal*.png")) + glob.glob(os.path.join(folder, "*normal*.exr"))
        norm_file = norm_candidates[0] if norm_candidates else None

        if valid_imgs:
            valid_imgs.sort(key=lambda p: (0 if "image" in os.path.basename(p).lower() else 1, len(p)))
            scenes.append({
                "scene_name": folder_name,
                "depth_path": d_path,
                "image_path": valid_imgs[0],
                "normal_path": norm_file,
            })
    return scenes

scenes = find_scenes(INPUT_DIR)
print(f"📦 Found {len(scenes)} scenes in {INPUT_DIR}")

# 2. Worker function executing native SPAG-4D pipeline with custom config


# 3. Multi-GPU Queue Dispatcher
work_q = queue.Queue()
for s in scenes:
    work_q.put(s)

results_lock = threading.Lock()
completed = []
failed = []

def gpu_worker(gpu_id):
    dev_str = f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu"
    while True:
        try:
            item = work_q.get_nowait()
        except queue.Empty:
            return

        name   = item["scene_name"]
        d_p    = item["depth_path"]
        i_p    = item["image_path"]
        norm_p = item.get("normal_path")

        scene_dir = os.path.join(OUTPUT_DIR, name)
        os.makedirs(scene_dir, exist_ok=True)
        out_ply = os.path.join(scene_dir, "splat.ply")
        dest_img = os.path.join(scene_dir, os.path.basename(i_p))

        t0 = time.time()
        print(f"[GPU {gpu_id}] ⏳ Processing: {name}")
        try:
            n_splats = process_scene_spag(d_p, i_p, out_ply, device=dev_str, normal_path=norm_p)
            if not os.path.exists(dest_img):
                shutil.copy2(i_p, dest_img)
            with results_lock:
                completed.append(name)
                print(f"[GPU {gpu_id}] ✅ Done: {name} ({n_splats:,} splats in {time.time()-t0:.2f}s)")
        except Exception as e:
            with results_lock:
                failed.append((name, str(e)))
                print(f"[GPU {gpu_id}] ❌ Failed: {name} -> {e}")

        work_q.task_done()

start = time.time()
threads = [threading.Thread(target=gpu_worker, args=(gid,)) for gid in GPU_IDS]
for t in threads: t.start()
for t in threads: t.join()

print(f"\n✨ Processing completed in {time.time()-start:.1f}s")
print(f"Completed: {len(completed)} | Failed: {len(failed)}")

📦 Found 2 scenes in /kaggle/input/datasets/newmailserver/moge-d2p-output/conference_room
[GPU 0] ⏳ Processing: 17-conference-room
[GPU 1] ⏳ Processing: 18-conference-room
[Outlier Pruning] Removed 9,147 floaters (strength=0.05)
[Outlier Pruning] Removed 11,422 floaters (strength=0.05)
Saved 515,141 Gaussians to /kaggle/working/output/17-conference-room/splat.ply (SH degree 0)
[GPU 0] ✅ Done: 17-conference-room (515,141 splats in 3.76s)
Saved 512,866 Gaussians to /kaggle/working/output/18-conference-room/splat.ply (SH degree 0)
[GPU 1] ✅ Done: 18-conference-room (512,866 splats in 3.81s)

✨ Processing completed in 4.1s
Completed: 2 | Failed: 0


In [4]:
# ============ UPLOAD TO KAGGLE (AUTO-VERSIONED) ============
import json, shutil, datetime, kagglehub

# Normalize into upload staging directory
if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)
os.makedirs(UPLOAD_DIR, exist_ok=True)

for scene_name in os.listdir(OUTPUT_DIR):
    src = os.path.join(OUTPUT_DIR, scene_name)
    dst = os.path.join(UPLOAD_DIR, scene_name)
    if os.path.isdir(src):
        shutil.copytree(src, dst)

print(f"📁 Staged {len(os.listdir(UPLOAD_DIR))} folders into {UPLOAD_DIR}")

# Dynamic timestamp for version tracking
current_time = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
version_note = f"Update run on {current_time} ({len(completed)} scenes processed)"

# Metadata
handle = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
metadata = {
    "title": "MoGe + SPAG-4D 3D Gaussian Splats",
    "id": handle,
    "licenses": [{"name": "CC0-1.0"}],
    "subtitle": "3D Gaussian Splatting PLY point clouds generated from MoGe depth and panorama images",
    "description": (
        f"3D Gaussian Splatting (.ply) models generated via SPAG-4D from MoGe depth maps.\n\n"
        f"- **Last Updated:** {current_time}\n"
        f"- **Total Scenes:** {len(completed)}\n"
        f"- **Completed Scenes:** {', '.join(completed)}"
    ),
    "keywords": ["gaussian-splatting", "3dgs", "moge", "spag4d", "panorama", "3d"]
}

with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

try:
    print(f"🚀 Uploading new version to Kaggle: {handle} ...")
    kagglehub.dataset_upload(handle, UPLOAD_DIR, version_notes=version_note)
    print(f"✅ SUCCESS: https://www.kaggle.com/datasets/{handle}")
except Exception as e:
    print(f"❌ Upload Failed: {e}")


📁 Staged 2 folders into /kaggle/temp/upload
🚀 Uploading new version to Kaggle: newmailserver/moge-splat-output ...
Uploading Dataset https://api.kaggle.com/datasets/newmailserver/moge-splat-output ...
Starting upload for file /kaggle/temp/upload/dataset-metadata.json

Uploading: 100%|██████████| 596/596 [00:00<00:00, 1.47kB/s]

Upload successful: /kaggle/temp/upload/dataset-metadata.json (596B)
Starting upload for file /kaggle/temp/upload/18-conference-room/splat.ply



Uploading: 100%|██████████| 34.9M/34.9M [00:01<00:00, 32.5MB/s]

Upload successful: /kaggle/temp/upload/18-conference-room/splat.ply (33MB)
Starting upload for file /kaggle/temp/upload/18-conference-room/image.jpg


...


Uploading: 100%|██████████| 35.0M/35.0M [00:00<00:00, 38.0MB/s]

Upload successful: /kaggle/temp/upload/17-conference-room/splat.ply (33MB)
Starting upload for file /kaggle/temp/upload/17-conference-room/image.jpg



Uploading: 100%|██████████| 589k/589k [00:00<00:00, 1.47MB/s]

Upload successful: /kaggle/temp/upload/17-conference-room/image.jpg (575KB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/newmailserver/moge-splat-output
✅ SUCCESS: https://www.kaggle.com/datasets/newmailserver/moge-splat-output
